# ⚡ CI/CD with FastAPI — Complete Beginner's Guide

---

## What You Will Learn

This notebook covers **CI/CD (Continuous Integration / Continuous Deployment)** using **FastAPI**, a modern Python web framework.

By the end of this notebook, you will understand:
1. What makes FastAPI different from Flask
2. How Python type hints and Pydantic models work
3. What async/await means and why it matters
4. How to test async FastAPI apps with httpx and pytest-asyncio
5. How a FastAPI CI/CD pipeline differs from Flask

---

## FastAPI vs Flask — The Big Picture

| | Flask | FastAPI |
|---|---|---|
| **Architecture** | WSGI (synchronous) | ASGI (async) |
| **Data validation** | Manual | Automatic via Pydantic |
| **API docs** | Need plugins | Auto-generated at /docs |
| **Type hints** | Optional | Required for models |
| **Testing** | Flask test client | httpx + pytest-asyncio |
| **Best for** | Beginners, small APIs | Production, ML APIs, high perf |

---

## What is Async / Await?

Imagine a restaurant kitchen:

**Synchronous (Flask)** = One chef:
```
Order 1: Take order → Cook soup (wait 5 min) → Serve soup → Take next order
Order 2: (must wait for Order 1 to finish)
```

**Asynchronous (FastAPI)** = One chef, multitasking:
```
Order 1: Take order → Start soup → (while soup cooks, handle Order 2)
Order 2: Take order → Start pasta → (pasta cooks, go back to soup)
Both orders finish much faster!
```

When your API waits for a database, network call, or file — async lets it handle other requests meanwhile. This is why FastAPI is much faster under load.

---

## What is Pydantic?

Pydantic is a Python library for **data validation using type hints**.

```python
# WITHOUT Pydantic (manual validation)
def create_task():
    data = request.get_json()
    if 'title' not in data:
        return error...
    if not isinstance(data['title'], str):
        return error...

# WITH Pydantic (automatic validation)
class TaskCreate(BaseModel):
    title: str          # Must be a string, required
    priority: str = 'low'  # Optional, defaults to 'low'
    done: bool = False  # Optional, defaults to False

# FastAPI automatically validates all of the above!
```

## Step 1: Install FastAPI and Testing Libraries

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================
# FastAPI needs more packages than Flask because it is async.
#
# requirements.txt for a FastAPI project:
# ───────────────────────────────────────
# fastapi==0.109.0       ← the web framework
# uvicorn==0.27.0        ← async server (like gunicorn but for ASGI)
# pydantic==2.5.0        ← data validation (bundled with fastapi)
# httpx==0.26.0          ← async HTTP client (used for testing)
# pytest==7.4.0          ← test runner
# pytest-asyncio==0.23.0 ← makes pytest work with async tests
# anyio==4.2.0           ← async compatibility layer
# ============================================================

!pip install fastapi uvicorn httpx pytest pytest-asyncio anyio --quiet

print('Packages installed!')
print()
print('Key packages explained:')
print('  fastapi       → The web framework')
print('  uvicorn       → ASGI server (runs the async app)')
print('  httpx         → Async HTTP client used in tests')
print('  pytest-asyncio → Makes pytest understand async def tests')

## Step 2: Build a FastAPI REST API

We build the same Task Manager API as the Flask notebook so you can directly compare them.

### Key FastAPI Concepts:
1. **Pydantic BaseModel** — defines the shape of request/response data
2. **async def** — defines an asynchronous route handler
3. **Type annotations** — FastAPI reads these to validate and document
4. **HTTPException** — raises HTTP errors with proper status codes

In [ ]:
# ============================================================
# STEP 2: Write the FastAPI Application
# ============================================================

fastapi_app_code = '''
# ============================================================
# fastapi_app.py — The FastAPI Application
# ============================================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

# ─────────────────────────────────────────────────────────────
# CREATE THE FASTAPI APP
# ─────────────────────────────────────────────────────────────
# The title and description appear in the auto-generated docs
# at http://localhost:8000/docs (Swagger UI)
# ─────────────────────────────────────────────────────────────
app = FastAPI(
    title="Task Manager API",
    description="A CI/CD demo API built with FastAPI",
    version="1.0.0"
)

# ─────────────────────────────────────────────────────────────
# PYDANTIC MODELS — Data Schemas
# ─────────────────────────────────────────────────────────────
# Pydantic models define the SHAPE of data.
# Think of them as strongly-typed data contracts.
#
# BaseModel provides:
#   - Automatic type validation (str, int, bool, etc.)
#   - Default values for optional fields
#   - JSON serialization/deserialization
#   - Auto-generated API documentation in /docs
# ─────────────────────────────────────────────────────────────

class TaskCreate(BaseModel):
    """
    The data a client sends when creating a new task (INPUT schema).
    FastAPI reads these type hints and:
      1. Validates the incoming JSON matches these types
      2. Returns a 422 error if validation fails — automatically!
      3. Shows this schema in the auto-generated API docs
    """
    title: str = Field(
        ...,             # ... means REQUIRED (no default value)
        min_length=1,    # Title cannot be empty
        max_length=200,  # Title cannot exceed 200 characters
        description="The task title"
    )
    done: bool = False      # Optional; defaults to False
    priority: str = "low"   # Optional; defaults to low


class Task(BaseModel):
    """
    The full task object returned by the API (OUTPUT schema).
    This is what the API sends back to the client.
    """
    id: int
    title: str
    done: bool
    priority: str


class TaskList(BaseModel):
    """Wrapper for returning a list of tasks with a total count."""
    tasks: List[Task]
    total: int


# ─────────────────────────────────────────────────────────────
# IN-MEMORY DATABASE
# ─────────────────────────────────────────────────────────────
tasks: dict = {}      # {task_id: Task}
next_id: list = [1]   # Auto-increment ID counter


# ─────────────────────────────────────────────────────────────
# ROUTE 1: Root Endpoint
# ─────────────────────────────────────────────────────────────
# FastAPI routes use same decorator pattern as Flask:
#   @app.get()    @app.post()    @app.delete()
#
# KEY DIFFERENCE: async def makes this an async function.
# FastAPI can handle other requests while awaiting I/O operations.
# ─────────────────────────────────────────────────────────────
@app.get("/")
async def root():
    """Welcome message."""
    return {
        "message": "Welcome to the Task Manager API",
        "version": "1.0.0",
        "framework": "FastAPI",
        "docs": "/docs"  # FastAPI auto-generates docs here!
    }


# ─────────────────────────────────────────────────────────────
# ROUTE 2: Health Check
# ─────────────────────────────────────────────────────────────
# Critical for CI/CD — load balancers check this endpoint.
# ─────────────────────────────────────────────────────────────
@app.get("/health")
async def health_check():
    """Health check for CI/CD monitoring."""
    return {"status": "healthy", "framework": "FastAPI"}


# ─────────────────────────────────────────────────────────────
# ROUTE 3: List all tasks
# ─────────────────────────────────────────────────────────────
# response_model=TaskList tells FastAPI:
#   - Validate the output matches TaskList schema
#   - Show TaskList in /docs as the response shape
# ─────────────────────────────────────────────────────────────
@app.get("/tasks", response_model=TaskList)
async def get_tasks():
    """Return all tasks."""
    task_list = list(tasks.values())
    return TaskList(tasks=task_list, total=len(task_list))


# ─────────────────────────────────────────────────────────────
# ROUTE 4: Create a task
# ─────────────────────────────────────────────────────────────
# HUGE DIFFERENCE FROM FLASK:
#   Flask:   data = request.get_json()  (manual, no validation)
#   FastAPI: def create_task(task_data: TaskCreate)
#
# FastAPI automatically:
#   1. Reads the request JSON body
#   2. Validates it matches TaskCreate schema
#   3. Returns 422 Unprocessable Entity if invalid
#   4. Passes the validated data as a TaskCreate object
# ─────────────────────────────────────────────────────────────
@app.post("/tasks", response_model=Task, status_code=201)
async def create_task(task_data: TaskCreate):
    """Create a new task. FastAPI validates the request body automatically."""
    task_id = next_id[0]
    next_id[0] += 1

    # task_data is already a validated TaskCreate object
    # No manual validation code needed!
    new_task = Task(
        id=task_id,
        title=task_data.title,       # Access fields with dot notation
        done=task_data.done,
        priority=task_data.priority
    )

    tasks[task_id] = new_task
    return new_task


# ─────────────────────────────────────────────────────────────
# ROUTE 5: Get a single task
# ─────────────────────────────────────────────────────────────
# task_id: int in the function signature tells FastAPI:
#   - Extract task_id from the URL path segment
#   - Convert it to an integer automatically
#   - If it cannot be an int, return 422 automatically
#
# HTTPException is FastAPI way to return HTTP error responses.
# raise HTTPException(status_code=404) returns a 404 JSON response.
# ─────────────────────────────────────────────────────────────
@app.get("/tasks/{task_id}", response_model=Task)
async def get_task(task_id: int):
    """Return a specific task by ID."""
    task = tasks.get(task_id)

    if task is None:
        raise HTTPException(
            status_code=404,
            detail=f"Task with id {task_id} not found"
        )

    return task


# ─────────────────────────────────────────────────────────────
# ROUTE 6: Delete a task
# ─────────────────────────────────────────────────────────────
@app.delete("/tasks/{task_id}")
async def delete_task(task_id: int):
    """Delete a task by ID."""
    task = tasks.pop(task_id, None)

    if task is None:
        raise HTTPException(
            status_code=404,
            detail=f"Task with id {task_id} not found"
        )

    return {"message": f"Task {task_id} deleted", "task": task}
'''

with open('fastapi_app.py', 'w') as f:
    f.write(fastapi_app_code)

print('fastapi_app.py created!')
print()
print('Key differences from Flask:')
print('  1. Pydantic models replace manual request validation')
print('  2. async def instead of def for route handlers')
print('  3. Type hints in function signatures drive validation')
print('  4. HTTPException replaces manual error returns')
print('  5. response_model= validates and documents the output')

## Step 3: Understand Pydantic Validation

Before testing, see how Pydantic catches bad data — this is what FastAPI uses internally.

In [ ]:
# ============================================================
# STEP 3: Pydantic Validation Demo
# ============================================================
# FastAPI calls Pydantic internally. Understanding it
# helps you debug validation errors and design better schemas.
# ============================================================

from pydantic import BaseModel, Field, ValidationError

class TaskCreate(BaseModel):
    title: str = Field(..., min_length=1, max_length=200)
    done: bool = False
    priority: str = 'low'


print('=' * 55)
print('DEMO 1: Valid task data')
print('=' * 55)
task = TaskCreate(title='Learn FastAPI', priority='high')
print(f'Result: {task}')
print(f'  task.title    = {task.title}')
print(f'  task.done     = {task.done}   <- default applied')
print(f'  task.priority = {task.priority}')

print()
print('=' * 55)
print('DEMO 2: Missing required field (no title)')
print('=' * 55)
try:
    bad = TaskCreate(priority='high')  # Missing title!
except ValidationError as e:
    print('ValidationError! FastAPI would return HTTP 422')
    for error in e.errors():
        print(f'  Field: {error["loc"]}')
        print(f'  Error: {error["msg"]}')

print()
print('=' * 55)
print('DEMO 3: min_length validation (empty title)')
print('=' * 55)
try:
    empty = TaskCreate(title='')  # Violates min_length=1
except ValidationError as e:
    print('ValidationError: empty title rejected')
    for error in e.errors():
        print(f'  {error["msg"]}')

print()
print('KEY POINT:')
print('  Flask  -> you write all this validation manually.')
print('  FastAPI -> Pydantic handles it automatically.')
print('  FastAPI returns HTTP 422 with field-level error detail.')

## Step 4: Write Async Tests

### Why Testing is Different from Flask

Flask uses a synchronous test client (`app.test_client()`).

FastAPI is async, so we need:
1. **httpx** — an async HTTP client
2. **pytest-asyncio** — lets pytest run `async def` test functions  
3. **ASGITransport** — connects httpx to our ASGI app without a server

```
Flask:   client = app.test_client()
         response = client.get('/tasks')

FastAPI: async with AsyncClient(transport=ASGITransport(app)) as client:
             response = await client.get('/tasks')
```

In [ ]:
# ============================================================
# STEP 4: Write Async Test File
# ============================================================

test_code = '''
# ============================================================
# test_fastapi_app.py — Async Tests for FastAPI Task Manager
# ============================================================
# Run with: pytest test_fastapi_app.py -v
#
# KEY DIFFERENCES FROM FLASK TESTS:
#   - Tests use async def instead of def
#   - Tests use await when making HTTP calls
#   - We use httpx.AsyncClient (not flask test_client)
#   - Validation errors return 422, not 400
# ============================================================

import pytest
import fastapi_app as fastapi_app_module
from httpx import AsyncClient, ASGITransport

# Tell pytest-asyncio to handle all async tests automatically
pytestmark = pytest.mark.asyncio


# ─────────────────────────────────────────────────────────────
# ASYNC FIXTURE: async_client
# ─────────────────────────────────────────────────────────────
# An async fixture runs async setup code before each test.
#
# ASGITransport connects httpx directly to our FastAPI ASGI app.
# No real server is started — pure in-process testing.
#
# base_url="http://test" is required but any string works.
# It is the base URL prepended to all routes.
# ─────────────────────────────────────────────────────────────
@pytest.fixture
async def async_client():
    """Create a fresh async HTTP client for each test."""
    app = fastapi_app_module.app

    # Reset state before each test for independence
    fastapi_app_module.tasks.clear()
    fastapi_app_module.next_id[0] = 1

    transport = ASGITransport(app=app)

    async with AsyncClient(transport=transport, base_url="http://test") as client:
        yield client  # Give the client to the test function


# ─────────────────────────────────────────────────────────────
# TEST 1: Root endpoint
# ─────────────────────────────────────────────────────────────
# await pauses execution until the HTTP request completes.
# While waiting, Python could handle other coroutines.
# ─────────────────────────────────────────────────────────────
async def test_root_endpoint(async_client):
    """Test root endpoint returns welcome message."""
    response = await async_client.get("/")  # await the async HTTP call

    assert response.status_code == 200
    data = response.json()   # httpx uses .json() not .get_json()
    assert data["framework"] == "FastAPI"
    assert "message" in data
    assert "docs" in data


# ─────────────────────────────────────────────────────────────
# TEST 2: Health check
# ─────────────────────────────────────────────────────────────
async def test_health_check(async_client):
    """Test health endpoint."""
    response = await async_client.get("/health")

    assert response.status_code == 200
    assert response.json()["status"] == "healthy"


# ─────────────────────────────────────────────────────────────
# TEST 3: Empty tasks list
# ─────────────────────────────────────────────────────────────
async def test_get_tasks_empty(async_client):
    """Fresh database returns empty list."""
    response = await async_client.get("/tasks")

    assert response.status_code == 200
    data = response.json()
    assert data["tasks"] == []
    assert data["total"] == 0


# ─────────────────────────────────────────────────────────────
# TEST 4: Create a task (happy path)
# ─────────────────────────────────────────────────────────────
async def test_create_task_success(async_client):
    """Creating a valid task returns 201 and task data."""
    task_data = {"title": "Write async tests", "priority": "high"}
    response = await async_client.post("/tasks", json=task_data)

    assert response.status_code == 201
    data = response.json()
    assert data["title"] == "Write async tests"
    assert data["priority"] == "high"
    assert data["done"] == False
    assert "id" in data


# ─────────────────────────────────────────────────────────────
# TEST 5: Pydantic validation — missing title
# ─────────────────────────────────────────────────────────────
# IMPORTANT DIFFERENCE FROM FLASK:
#   Flask returns 400 Bad Request for validation errors.
#   FastAPI returns 422 Unprocessable Entity (Pydantic generated).
#   422 means: understood the request but data is invalid.
# ─────────────────────────────────────────────────────────────
async def test_create_task_missing_title(async_client):
    """Missing title returns 422 Unprocessable Entity."""
    response = await async_client.post("/tasks", json={"priority": "high"})

    assert response.status_code == 422  # NOT 400! FastAPI uses 422
    assert "detail" in response.json()  # FastAPI error responses use detail


# ─────────────────────────────────────────────────────────────
# TEST 6: Pydantic validation — empty title
# ─────────────────────────────────────────────────────────────
async def test_create_task_empty_title(async_client):
    """Empty title violates min_length=1 and returns 422."""
    response = await async_client.post("/tasks", json={"title": ""})
    assert response.status_code == 422


# ─────────────────────────────────────────────────────────────
# TEST 7: Get a specific task
# ─────────────────────────────────────────────────────────────
async def test_get_specific_task(async_client):
    """Retrieve a task by ID."""
    create_response = await async_client.post(
        "/tasks", json={"title": "Deploy with CI/CD"}
    )
    task_id = create_response.json()["id"]

    get_response = await async_client.get(f"/tasks/{task_id}")

    assert get_response.status_code == 200
    assert get_response.json()["title"] == "Deploy with CI/CD"


# ─────────────────────────────────────────────────────────────
# TEST 8: 404 for non-existent task
# ─────────────────────────────────────────────────────────────
async def test_get_nonexistent_task(async_client):
    """Non-existent task returns 404."""
    response = await async_client.get("/tasks/9999")

    assert response.status_code == 404
    assert "detail" in response.json()


# ─────────────────────────────────────────────────────────────
# TEST 9: Delete a task
# ─────────────────────────────────────────────────────────────
async def test_delete_task(async_client):
    """Deleting a task removes it from the database."""
    create_response = await async_client.post("/tasks", json={"title": "To delete"})
    task_id = create_response.json()["id"]

    await async_client.delete(f"/tasks/{task_id}")

    get_response = await async_client.get(f"/tasks/{task_id}")
    assert get_response.status_code == 404


# ─────────────────────────────────────────────────────────────
# TEST 10: Full workflow integration test
# ─────────────────────────────────────────────────────────────
async def test_full_workflow(async_client):
    """Integration: create tasks, list them, delete one."""
    titles = ["Set up pipeline", "Write tests", "Ship it"]
    created_ids = []

    for title in titles:
        response = await async_client.post("/tasks", json={"title": title})
        created_ids.append(response.json()["id"])

    list_response = await async_client.get("/tasks")
    assert list_response.json()["total"] == 3

    await async_client.delete(f"/tasks/{created_ids[0]}")

    list_response = await async_client.get("/tasks")
    assert list_response.json()["total"] == 2
'''

with open('test_fastapi_app.py', 'w') as f:
    f.write(test_code)

print('test_fastapi_app.py created!')
print()
print('Key differences from Flask tests:')
print('  1. Tests are async def functions')
print('  2. HTTP calls use await')
print('  3. Uses httpx.AsyncClient + ASGITransport')
print('  4. Validation errors return 422, not 400')
print('  5. Error responses use detail field, not error field')

## Step 5: Run the Async Tests

In [ ]:
# ============================================================
# STEP 5: Run pytest — The CI Pipeline Test Stage
# ============================================================
# In GitHub Actions the YAML test step looks identical to Flask:
#
#   - name: Run tests
#     run: pytest test_fastapi_app.py -v
#
# pytest-asyncio handles the async complexity internally.
# The pipeline does not know or care that tests are async.
# ============================================================

# Write pytest.ini to enable automatic async mode
with open('pytest.ini', 'w') as f:
    f.write('[pytest]\nasyncio_mode = auto\n')

print('Running the FastAPI CI/CD test stage...')
print('Command: pytest test_fastapi_app.py -v')
print('=' * 60)

import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'test_fastapi_app.py', '-v', '--tb=short'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print('=' * 60)
if result.returncode == 0:
    print('ALL TESTS PASSED — Pipeline would proceed to deployment!')
else:
    print('TESTS FAILED — Pipeline would STOP here.')

## Step 6: FastAPI CI/CD Pipeline + Dockerfile

In [ ]:
# ============================================================
# STEP 6: GitHub Actions YAML and Dockerfile for FastAPI
# ============================================================

fastapi_yaml = """
# ============================================================
# .github/workflows/ci.yml — FastAPI CI/CD Pipeline
# ============================================================
# The pipeline STRUCTURE is identical to Flask.
# Differences: requirements.txt packages and CMD in Dockerfile.
# ============================================================

name: FastAPI CI/CD Pipeline

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    name: Build and Test
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v4
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install -r requirements.txt
        # requirements.txt: fastapi, uvicorn, httpx, pytest, pytest-asyncio, anyio

      - name: Run tests
        run: pytest test_fastapi_app.py -v --tb=short
        # pytest-asyncio handles async def tests automatically

  deploy:
    name: Deploy
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == 'refs/heads/main'

    steps:
      - uses: actions/checkout@v4
      - name: Build and push Docker image
        run: |
          docker build -t fastapi-task-api:latest .
          docker push myusername/fastapi-task-api:latest
"""

fastapi_dockerfile = """
# ============================================================
# Dockerfile for FastAPI
# ============================================================
# Almost identical to Flask's Dockerfile.
# THE ONLY DIFFERENCE: CMD uses uvicorn instead of gunicorn.
#
# uvicorn = ASGI server (for async frameworks like FastAPI)
# gunicorn = WSGI server (for sync frameworks like Flask)
# ============================================================

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

# uvicorn replaces gunicorn for ASGI/async apps
# --host 0.0.0.0      → listen on all network interfaces
# --port 8000         → FastAPI convention (Flask uses 5000)
# --workers 4         → 4 parallel async workers
# fastapi_app:app     → file fastapi_app.py, object named app
CMD ["uvicorn", "fastapi_app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
"""

print('GitHub Actions YAML (FastAPI):')
print(fastapi_yaml)
print()
print('Dockerfile (FastAPI):')
print(fastapi_dockerfile)
print()
print('Flask vs FastAPI production servers:')
print('  Flask:   gunicorn --workers 4 --bind 0.0.0.0:5000 app:app')
print('  FastAPI: uvicorn fastapi_app:app --host 0.0.0.0 --port 8000')

In [ ]:
# ============================================================
# STEP 7: Final Comparison Summary
# ============================================================

print("""
Flask vs FastAPI — Side-by-Side Code Summary
============================================

CREATING THE APP
  Flask:    app = Flask(__name__)
  FastAPI:  app = FastAPI(title="My API", version="1.0.0")

DEFINING A ROUTE
  Flask:    @app.route('/tasks', methods=['GET'])
            def get_tasks():
                return jsonify({...})

  FastAPI:  @app.get('/tasks')
            async def get_tasks():
                return {...}  # auto-serialized

READING REQUEST DATA
  Flask:    data = request.get_json()   # manual, no validation
            title = data.get('title')

  FastAPI:  async def create(task: TaskCreate):
            # task is already validated by Pydantic

RETURNING ERRORS
  Flask:    return jsonify({"error": "..."}), 404
  FastAPI:  raise HTTPException(status_code=404, detail="...")

VALIDATION ERRORS
  Flask:    Manual -> returns 400 Bad Request
  FastAPI:  Automatic -> returns 422 Unprocessable Entity

TESTING
  Flask:    client = app.test_client()
            response = client.get('/tasks')

  FastAPI:  async with AsyncClient(...) as client:
                response = await client.get('/tasks')

PRODUCTION SERVER
  Flask:    gunicorn (WSGI — synchronous)
  FastAPI:  uvicorn  (ASGI — asynchronous)

CI/CD PIPELINE
  Both:     Identical pipeline structure (GitHub Actions, pytest, Docker)
  Diff:     Only requirements.txt contents and Dockerfile CMD

WHEN TO CHOOSE
  Flask:   Learning APIs, small projects, synchronous work
  FastAPI: Production APIs, ML model serving, high concurrency
""")

print('Congratulations! FastAPI CI/CD notebook complete.')
print()
print('Challenge: Add a PUT /tasks/{task_id} endpoint')
print('  1. Create a TaskUpdate Pydantic model')
print('  2. Write an async PUT route handler')
print('  3. Write a test: async def test_update_task(async_client)')